# Run Gleason-siloed survival pipeline locally

Builds and runs the **Gleason-siloed** analysis for **both** anchors (first-treatment and first-ARPI). Each run is restricted to the sub-cohort with an ISUP Gleason grade group on/before the anchor, and adds `GLEASON_GROUP` (ordinal 1-5, closest record on/before the anchor) as a covariate in the univariate Cox associations, the multivariable Cox model, and XGBoost. PGS is not run.

Outputs go to separate `*_gleason` trees so nothing clobbers the base pipelines. Run the cells top to bottom (preprocessing -> build x2 -> models x2 -> inspect).

## Paths

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis" / "PROFILE"

NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/")
DATA           = NEPC_PROJ_PATH / "longitudinal_prediction_data.csv"   # output of preprocessing
GLEASON_FILE   = NEPC_PROJ_PATH / "LLM_gleason_timeline" / "gleason_timeline.tsv"

SURV = NEPC_PROJ_PATH / "survival_analysis" / "PROFILE"
# (anchor label, build --anchor-col, inputs dir, outputs dir)
ANCHORS = [
    ("first-treatment", "t_first_treatment",  SURV / "prediction_inputs_gleason",           SURV / "local_runs_gleason"),
    ("first-ARPI",      "t_treatment_anchor", SURV / "prediction_inputs_tx_anchor_gleason", SURV / "local_runs_tx_anchor_gleason"),
]

os.chdir(PROJECT_ROOT)
for _, _, idir, odir in ANCHORS:
    idir.mkdir(parents=True, exist_ok=True)
    odir.mkdir(parents=True, exist_ok=True)

PYTHON = sys.executable
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"

print("python:      ", PYTHON)
print("data:        ", DATA)
print("gleason_file:", GLEASON_FILE)
for label, anchor, idir, odir in ANCHORS:
    print(f"  {label:16s} anchor={anchor:18s} -> {idir.name} / {odir.name}")

## 1. Preprocess raw labs

Re-runs `longitudinal_data_processing.py` so `longitudinal_prediction_data.csv` carries `t_treatment_anchor` (needed for the first-ARPI build). Skip if already current.

In [ ]:
!{PYTHON} {SURVIVAL_DIR}/longitudinal_data_processing.py

## 2. Build Gleason-augmented prediction inputs (both anchors)

Each build adds a `GLEASON_GROUP` column (closest ISUP grade group on/before the anchor) to every `aggregated_landmark{D}.csv`. Pass `--gleason-group-col` / `--gleason-date-col` if the tsv column names aren't auto-detected.

In [ ]:
import subprocess, time

for label, anchor, idir, odir in ANCHORS:
    cmd = [
        PYTHON, str(SURVIVAL_DIR / "build_prediction_inputs.py"),
        "--data", str(DATA),
        "--output-dir", str(idir),
        "--anchor-col", anchor,
        "--gleason-file", str(GLEASON_FILE),
        "--landmark-days", "0", "90",
        "--time-unit-days", "7",
        "--test-frac", "0.20", "--val-frac", "0.20",
        "--min-patient-coverage", "0.20",
    ]
    print(f"[build] {label}\n        " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    print(f"[done ] {label} -> rc={rc} ({(time.time()-t0)/60:.1f} min)\n")

## 3. Run the Gleason-siloed models (both anchors)

For each anchor: cox univariate + multivariable (both/baseline) + xgboost (both/baseline), all with `--with-gleason --restrict-to-gleason`. PGS is intentionally not run.

In [ ]:
import subprocess, time

N_FOLDS = 5
GLEASON_FLAGS = ["--with-gleason", "--restrict-to-gleason"]

# (model, landmark, config_dir)
TASKS = [
    ("cox_univariate",    0,  "both"),
    ("cox_univariate",    90, "both"),
    ("cox_multivariable", 0,  "both"),
    ("cox_multivariable", 90, "both"),
    ("cox_multivariable", 0,  "baseline"),
    ("cox_multivariable", 90, "baseline"),
    ("xgboost",           0,  "both"),
    ("xgboost",           90, "both"),
    ("xgboost",           0,  "baseline"),
    ("xgboost",           90, "baseline"),
]


def build_command(model, landmark, config_dir, inputs_dir, row_output_dir):
    if model == "cox_univariate":
        return [PYTHON, str(SURVIVAL_DIR / "cox_univariate.py"),
                "--inputs-dir", str(inputs_dir), "--output-dir", str(row_output_dir),
                "--landmark-days", str(landmark), "--endpoints", "platinum", *GLEASON_FLAGS]
    if model == "cox_multivariable":
        cmd = [PYTHON, str(SURVIVAL_DIR / "cox_multivariable.py"),
               "--inputs-dir", str(inputs_dir), "--output-dir", str(row_output_dir),
               "--landmark-days", str(landmark), "--endpoints", "platinum",
               "--n-folds", str(N_FOLDS), *GLEASON_FLAGS]
        if config_dir == "baseline":
            cmd.append("--baseline")
        return cmd
    if model == "xgboost":
        cmd = [PYTHON, str(SURVIVAL_DIR / "landmark_xgboost.py"),
               "--inputs-dir", str(inputs_dir), "--output-dir", str(row_output_dir),
               "--landmark-days", str(landmark), "--endpoints", "platinum",
               "--n-folds", str(N_FOLDS), *GLEASON_FLAGS]
        if config_dir == "baseline":
            cmd.append("--baseline")
        return cmd
    raise ValueError(model)


summary = []
for label, anchor, inputs_dir, output_dir in ANCHORS:
    print(f"\n========== {label} (Gleason-siloed) ==========")
    for model, landmark, config_dir in TASKS:
        model_dir = "cox" if model in ("cox_univariate", "cox_multivariable") else model
        row_output_dir = output_dir / model_dir / f"landmark_{landmark}" / config_dir
        row_output_dir.mkdir(parents=True, exist_ok=True)
        cmd = build_command(model, landmark, config_dir, inputs_dir, row_output_dir)
        tag = f"{label:16s} {model:18s} landmark_{landmark:<2} {config_dir}"
        print(f"[run ] {tag}")
        t0 = time.time()
        rc = subprocess.call(cmd)
        status = "ok" if rc == 0 else f"FAILED (rc={rc})"
        print(f"[done] {tag} -> {status} ({(time.time()-t0)/60:.1f} min)\n")
        summary.append((tag, status))

print("\n=== Gleason run summary ===")
for tag, status in summary:
    print(f"  {tag}  {status}")

## 4. Inspect the Gleason associations

Pulls the GLEASON_GROUP univariate row (HR per grade-group increase, p, q) for each anchor and landmark.

In [ ]:
import pandas as pd

rows = []
for label, anchor, inputs_dir, output_dir in ANCHORS:
    for landmark in (0, 90):
        p = output_dir / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
        if not p.exists():
            continue
        df = pd.read_csv(p)
        g = df.loc[(df["feature"] == "GLEASON_GROUP") & (df["endpoint"] == "platinum")]
        if g.empty:
            continue
        r = g.iloc[0]
        rows.append({
            "anchor": label, "landmark": landmark,
            "hazard_ratio_per_grade": r.get("hazard_ratio_per_sd"),
            "coef": r.get("coef_feature"), "p_value": r.get("p_value"),
            "q_value": r.get("q_value"), "n_used": r.get("n_patients_used"),
        })

gleason_assoc = pd.DataFrame(rows)
gleason_assoc